# Extract Data From the Customer JSON File

1. Query Single File
2. Query List of Files using wildcard Characters
3. Query all the files in a Folder
4. Select File Metadata
5. Create Table in Bronze Schema

#### 1. Query Single Json File

[PySpark](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.format.html)

We can use:

```
df = spark.read.format("json").load(path)
```

or 
```
df = spark.read.json(path)
```


In [0]:
%fs ls /Volumes/gizmobox/landing/operational_data/customers

In [0]:
%python
df = spark.read.format("json").load("/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json")
display(df)


#### 2. Query Multiple Json Files


In [0]:
%python
df = spark.read.format("json").load("/Volumes/gizmobox/landing/operational_data/customers/customers_2024_*.json")
display(df)

#### 3. Query all the files in a folder


In [0]:
%python
# SELECT * FROM json.`/Volumes/gizmobox/landing/operational_data/customers/`
# df = spark.read.format("json").load("/Volumes/gizmobox/landing/operational_data/customers/")
df = spark.read.json("/Volumes/gizmobox/landing/operational_data/customers/")
display(df)

### 4. Select File Metadata

[DataFrame.select](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.select.html)

In [0]:
%python
# SELECT 
#     _metadata.file_path as file_path,
#     * 
# FROM json.`/Volumes/gizmobox/landing/operational_data/customers/`
df_with_metadata = df.select("_metadata.file_path", "*")
display(df_with_metadata)



### 5. Create Table in the bronze schema

[PySpark writer](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.format.html)

- Recommended DataFrameWriterV2

In [0]:
%python
# CREATE OR REPLACE VIEW gizmobox.bronze.v_customers 
# AS
# SELECT 
#     *,
#     _metadata.file_path as file_path
# FROM json.`/Volumes/gizmobox/landing/operational_data/customers/`

'''
with v1
'''
df_with_metadata.write.format("delta").mode("overwrite").saveAsTable("gizmobox.bronze.py_customers")

In [0]:
'''
With v2
'''

df_with_metadata.writeTo("gizmobox.bronze.py_customers").createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.bronze.py_customers;